# Cobertura de semilla: el cuello de botella

El 61 % de las pseudohuérfanas tiene **semilla nula**: ningún vecino químico con
blanco conocido. Mientras eso siga así ninguna reponderación mueve el número —las
iteraciones it0–it3 de v3 dieron resultados idénticos, que es exactamente lo que
se espera si la limitación es estructural.

Este notebook mide el techo de las tres palancas, sin re-correr el modelo:

  * **umbral de similitud química** — la capa guardada sólo conserva pares con
    similitud ≥ 0.8, así que desde acá sólo se puede subir; bajarlo exige
    recalcular sobre los fingerprints crudos (`reproducir_v6/cobertura_semilla/01`);
  * **KEGG** como tercera fuente de afiliaciones — rescata semillas *no
    informativas*, no nulas: agrega categorías, no vecinos;
  * **capa fenotípica** como semilla de respaldo — no da un blanco proteico, da
    una especie; es la información más débil, que el paper descarta explícitamente.

Fuente: `reproducir_v6/cobertura_semilla/`.
Salidas: `02_cobertura_por_droga.csv`, `02_cobertura_por_especie.csv`, `02_palancas.csv`.

## Imports

In [1]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

V4 = "/home/ggiordano/TDR/TDR_2026_v4"
sys.path.insert(0, f"{V4}/comun")
sys.path.insert(0, f"{V4}/huerfanas")
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_huerfanas as fh              # el .py de esta carpeta

SALIDAS = tdr.out("huerfanas")
FIGURAS = SALIDAS / "figuras"
NB      = "02"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

PosixPath('/home/ggiordano/TDR/TDR_2026_v4/gon4/huerfanas_out')

## Datos

In [2]:
# cluster_consistent=True: `huerfanas/` filtra los positivos/negativos
# inconsistentes a nivel cluster, como orphan_drugs_v4.ipynb.
crudo = tdr.cargar_db(anotaciones=True, quimica=True, fenotipo=True, cluster_consistent=True)

# README §7.3: el filtro de promiscuidad se APLICA antes de armar ninguna semilla.
# La lista sale de analiceDB/03; si falta, el error dice qué correr.
promiscuos = tdr.compuestos_promiscuos()
datos = tdr.filtrar_capa_quimica(crudo, promiscuos)

HUERFANAS = tdr.out("huerfanas")
res01 = fh.cargar_resultados(HUERFANAS, "01", patron="*_pseudohuerfanas")
res01["semilla"].value_counts(normalize=True)

· 00_specie_target
· 03_bioactivities_target_compound
· 04a_interpro / 04b_orthomcl
· 01/02 clusters y aristas quimicas (pesado)
· 03_bioactivities_organism_compound
filtro de promiscuidad (MW < 150 y N_parentales > 100): 25 compuestos
  sclus 2396103 -> 2396078 (0.00 % removido)
  dds   973 -> 973 (0.00 % removido)


semilla
nula              0.831406
no informativa    0.141994
informativa       0.026600
Name: proportion, dtype: float64

## Acondicionamiento

In [3]:
k1, posdt_sp = fh.construir_pseudohuerfanas(datos)
# La palanca se evalúa sobre las que hoy NO son informativas: es el universo a rescatar
a_rescatar = res01.loc[res01["semilla"] != "informativa", ["drug_id", "especie"]]
muestra = (k1.merge(a_rescatar.rename(columns={"especie": "sp_id"}), on=["drug_id", "sp_id"])
           .pipe(fh.muestra_pseudohuerfanas, n=500))
print(f"{len(muestra)} drogas a rescatar, de {len(res01)} evaluadas en el notebook 01")
muestra["grupo"].value_counts().sort_index()

pseudohuerfanas: 60253 drogas, 26 especies, grupos {0: 22375, 1: 17306, 2: 9830, 3: 10742}
500 drogas a rescatar, de 7782 evaluadas en el notebook 01


grupo
0    176
1     86
2    132
3    106
Name: count, dtype: int64

## Corrida

In [4]:
# celda de corrida: el ciclo se lee aca
palancas = []
for palanca, kw in [("umbral", {"umbral": 0.9}),
                    ("kegg", {"mapa_kegg": None}),      # pasar el mapeo si se consiguió
                    ("fenotipo", {})]:
    palancas.append(fh.cobertura_por_palanca(datos, palanca, muestra, **kw))
pal = pd.concat(palancas, ignore_index=True)

cob_esp = (res01.groupby(["especie", "semilla"]).size().unstack(fill_value=0)
           .assign(techo=lambda d: d.get("informativa", 0) / d.sum(axis=1)))

pal.to_csv(SALIDAS / f"{NB}_palancas.csv", index=False)
res01[["drug_id", "especie", "semilla", "n_semilla"]].to_csv(
    SALIDAS / f"{NB}_cobertura_por_droga.csv", index=False)
cob_esp.to_csv(SALIDAS / f"{NB}_cobertura_por_especie.csv")
fh.escribir_meta(SALIDAS, NB, notebook="02_cobertura_semilla.ipynb",
                 params={"umbral": 0.9, "n_muestra": len(muestra)},
                 fuente_nb="01", filtro_promiscuidad=True)

palanca umbral: rescata 0 de 500 (0.0%)


KeyboardInterrupt: 

# Resultados

In [ ]:
pal     = pd.read_csv(SALIDAS / f"{NB}_palancas.csv")
cob_esp = pd.read_csv(SALIDAS / f"{NB}_cobertura_por_especie.csv", index_col=0)
cob_esp.round(3)

In [ ]:
fig = fh.fig_cobertura(res01, plt)
tdr.guardar(fig, f"{NB}_f01_descomposicion", FIGURAS)

In [ ]:
fig = fh.fig_palancas(pal, plt)
tdr.guardar(fig, f"{NB}_f02_palancas", FIGURAS)

In [ ]:
# Techo teórico: ninguna reponderación puede superar la fracción informativa
fig, ax = plt.subplots(figsize=(6.5, 3.5), tight_layout=True)
t = cob_esp.sort_values("techo")
ax.barh(range(len(t)), 100 * t["techo"], color=tdr.S3)
ax.set_yticks(range(len(t)))
ax.set_yticklabels([tdr.NOMBRE_CORTO.get(s, s) for s in t.index], fontsize=7)
ax.set_xlabel("% con semilla informativa (techo del método)")
tdr.guardar(fig, f"{NB}_f03_techo_teorico", FIGURAS)